# Análise e Previsão de Demanda — E-commerce

## 1. Objetivo do projeto

Este notebook integra um projeto de análise de dados e otimização logística de um e-commerce, utilizando dados históricos da Olist (2016–2018).

O objetivo é analisar o comportamento da demanda, avaliar modelos de previsão e apoiar a simulação de cenários de expansão logística.

## 2. Tecnologias utilizadas

- Python
- Pandas
- PostgreSQL
- SQLAlchemy
- Scikit-learn
- Power BI

## 3. Metodologia

O desenvolvimento contempla:

1. Extração dos dados históricos armazenados no PostgreSQL.
2. Análise da evolução mensal dos pedidos nos estados de São Paulo, Rio de Janeiro e Minas Gerais.
3. Separação dos dados em períodos de treinamento e teste.
4. Avaliação de um modelo de regressão linear.
5. Comparação com um modelo de referência (baseline).
6. Construção de cenários hipotéticos de crescimento da demanda.
7. Simulação dos custos logísticos de diferentes localizações de centros de distribuição.

## 4. Observação metodológica

Os dados representam o histórico da plataforma Olist, não o mercado brasileiro de comércio eletrônico como um todo.

Como o modelo de regressão linear não superou o baseline nos testes realizados, os cenários de expansão foram construídos a partir de hipóteses explícitas de crescimento, e não de previsões consideradas confiáveis.

Os custos logísticos são simulados e não representam tarifas reais de transporte.

In [ ]:
print("Python funcionando! Vamos começar o Machine Learning.")

In [ ]:
%pip install pandas sqlalchemy psycopg2-binary

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import psycopg2

print("Bibliotecas importadas com sucesso!")

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

senha = getpass("Digite a senha do PostgreSQL: ")

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=senha,
    host="localhost",
    port=5432,
    database="ecommerce_expansao"
)

engine = create_engine(url)

with engine.connect() as conexao:
    resultado = conexao.execute(text("SELECT 1"))
    print("Conexão realizada com sucesso!")
    

In [ ]:
import pandas as pd

query = """
SELECT
    estado,
    mes,
    total_pedidos,
    periodo,
    indice_mes
FROM raw.demanda_treino_teste
ORDER BY estado, mes;
"""

df = pd.read_sql(query, engine)

print("Quantidade de linhas:", len(df))
print("\nPrimeiros registros:")
display(df.head())

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Separar treinamento e teste
treino = df[df["periodo"] == "treino"].copy()
teste = df[df["periodo"] == "teste"].copy()

# Criar uma coluna para armazenar previsões
teste["previsao"] = 0.0

# Treinar um modelo para cada estado
for estado in ["SP", "RJ", "MG"]:

    dados_treino = treino[treino["estado"] == estado]
    dados_teste = teste[teste["estado"] == estado]

    modelo = LinearRegression()

    modelo.fit(
        dados_treino[["indice_mes"]],
        dados_treino["total_pedidos"]
    )

    previsoes = modelo.predict(
        dados_teste[["indice_mes"]]
    )

    teste.loc[dados_teste.index, "previsao"] = previsoes

# Comparar valores reais e previstos
teste["erro_absoluto"] = abs(
    teste["total_pedidos"] - teste["previsao"]
)

display(
    teste[[
        "estado",
        "mes",
        "total_pedidos",
        "previsao",
        "erro_absoluto"
    ]]
)

print(
    "Erro médio absoluto:",
    round(
        mean_absolute_error(
            teste["total_pedidos"],
            teste["previsao"]
        ),
        2
    )
)

In [ ]:
from sklearn.metrics import mean_absolute_error

# Organizar os dados por estado e mês
dados = df.sort_values(["estado", "mes"]).copy()

# Previsão usando os pedidos do mês anterior
dados["previsao_simples"] = (
    dados.groupby("estado")["total_pedidos"].shift(1)
)

# Selecionar apenas os meses de teste
teste_simples = dados[dados["periodo"] == "teste"].copy()

# Calcular o erro da previsão simples
erro_simples = mean_absolute_error(
    teste_simples["total_pedidos"],
    teste_simples["previsao_simples"]
)

# Comparar os dois modelos
erro_regressao = mean_absolute_error(
    teste["total_pedidos"],
    teste["previsao"]
)

print("MAE Regressão Linear:", round(erro_regressao, 2))
print("MAE Previsão Simples:", round(erro_simples, 2))

## Avaliação dos modelos de previsão de demanda

### Objetivo

Avaliar se um modelo de regressão linear consegue prever a
quantidade mensal de pedidos com maior precisão do que um
modelo de referência simples (baseline).

### Metodologia

Foram analisados os pedidos históricos da plataforma Olist
nos estados de São Paulo (SP), Rio de Janeiro (RJ) e
Minas Gerais (MG).

Os dados foram divididos cronologicamente:

- Treinamento: janeiro de 2017 a abril de 2018 (16 meses).
- Teste: maio de 2018 a agosto de 2018 (4 meses).

Foram comparadas duas abordagens:

1. Regressão Linear: utiliza o índice temporal do mês para
   estimar a quantidade de pedidos.

2. Baseline: utiliza a quantidade de pedidos do mês anterior
   como previsão para o mês seguinte.

A avaliação foi realizada utilizando o MAE
(Mean Absolute Error), que representa o erro absoluto médio.
Valores menores indicam previsões mais próximas dos
valores observados.

### Resultados

| Estado | MAE Regressão Linear | MAE Baseline |
|--------|--------------------:|-------------:|
| SP     | 673,83              | 212,25       |
| RJ     | 360,38              | 76,00        |
| MG     | 318,68              | 75,75        |

### Interpretação

O baseline apresentou menor erro médio nos três estados
analisados.

Isso indica que, para o período de teste utilizado,
a regressão linear não ofereceu melhoria em relação
ao modelo de referência.

O desempenho observado não sustenta a utilização
da regressão linear como ferramenta confiável de
projeção de demanda futura neste projeto.

### Decisão metodológica

Em vez de utilizar previsões de Machine Learning
sem desempenho satisfatório, foram definidos três
cenários hipotéticos de crescimento:

- Conservador
- Moderado
- Acelerado

Esses cenários permitem avaliar como diferentes
hipóteses de demanda afetam os custos logísticos
e a escolha da localização do centro de distribuição.

Os cenários são simulações, não previsões estatísticas.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Comparar os modelos por estado
resultados = []

for estado in ["SP", "RJ", "MG"]:

    # Regressão Linear
    dados_regressao = teste[teste["estado"] == estado]

    mae_regressao = mean_absolute_error(
        dados_regressao["total_pedidos"],
        dados_regressao["previsao"]
    )

    # Previsão Simples
    dados_simples = teste_simples[
        teste_simples["estado"] == estado
    ]

    mae_simples = mean_absolute_error(
        dados_simples["total_pedidos"],
        dados_simples["previsao_simples"]
    )

    resultados.append({
        "Estado": estado,
        "MAE Regressão Linear": round(mae_regressao, 2),
        "MAE Previsão Simples": round(mae_simples, 2)
    })

comparacao = pd.DataFrame(resultados)

display(comparacao)

In [ ]:
from sklearn.metrics import mean_absolute_error
import pandas as pd

# Criar uma previsão simples sem usar dados do teste
teste_justo = teste.copy()

# Último mês disponível no treinamento
ultimo_mes = (
    treino.sort_values("mes")
    .groupby("estado")["total_pedidos"]
    .last()
)

# Prever todos os meses de teste
# usando o último valor conhecido
teste_justo["previsao_simples"] = (
    teste_justo["estado"].map(ultimo_mes)
)

# Comparar os modelos por estado
resultados = []

for estado in ["SP", "RJ", "MG"]:

    dados = teste_justo[
        teste_justo["estado"] == estado
    ]

    mae_regressao = mean_absolute_error(
        dados["total_pedidos"],
        dados["previsao"]
    )

    mae_simples = mean_absolute_error(
        dados["total_pedidos"],
        dados["previsao_simples"]
    )

    resultados.append({
        "Estado": estado,
        "MAE Regressão Linear": round(mae_regressao, 2),
        "MAE Previsão Simples": round(mae_simples, 2)
    })

comparacao_justa = pd.DataFrame(resultados)

display(comparacao_justa)

In [ ]:
import matplotlib.pyplot as plt

estado = "SP"

dados = teste_justo[
    teste_justo["estado"] == estado
].sort_values("mes")

plt.figure(figsize=(10, 5))

plt.plot(
    dados["mes"],
    dados["total_pedidos"],
    marker="o",
    label="Pedidos reais"
)

plt.plot(
    dados["mes"],
    dados["previsao"],
    marker="o",
    label="Regressão Linear"
)

plt.plot(
    dados["mes"],
    dados["previsao_simples"],
    marker="o",
    label="Previsão Simples"
)

plt.title("Previsão de Demanda — São Paulo")
plt.xlabel("Mês")
plt.ylabel("Quantidade de pedidos")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

for estado in ["SP", "RJ", "MG"]:
    dados_estado = df[
        df["estado"] == estado
    ].sort_values("mes")

    plt.plot(
        dados_estado["mes"],
        dados_estado["total_pedidos"],
        marker="o",
        label=estado
    )

plt.title("Evolução Mensal da Demanda por Estado")
plt.xlabel("Mês")
plt.ylabel("Quantidade de pedidos")
plt.legend(title="Estado")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd

dados = df.copy()

dados["mes"] = pd.to_datetime(dados["mes"])
dados["ano"] = dados["mes"].dt.year
dados["numero_mes"] = dados["mes"].dt.month

# Comparar os mesmos meses dos dois anos
comparacao = dados[
    dados["numero_mes"].between(1, 8)
]

resumo = (
    comparacao
    .groupby(["estado", "ano"])["total_pedidos"]
    .sum()
    .unstack()
)

resumo["crescimento_percentual"] = (
    (resumo[2018] - resumo[2017])
    / resumo[2017]
) * 100

display(resumo.round(2))

In [ ]:
import pandas as pd

# Demanda histórica de janeiro a agosto de 2018
demanda_base = {
    "SP": 23335,
    "RJ": 6342,
    "MG": 6079
}

# Cenários hipotéticos de crescimento
cenarios = {
    "Conservador": {
        "SP": 0.05,
        "RJ": 0.05,
        "MG": 0.05
    },
    "Moderado": {
        "SP": 0.15,
        "RJ": 0.10,
        "MG": 0.12
    },
    "Acelerado": {
        "SP": 0.30,
        "RJ": 0.20,
        "MG": 0.25
    }
}

resultados = []

for cenario, taxas in cenarios.items():
    for estado, crescimento in taxas.items():

        demanda_projetada = (
            demanda_base[estado] * (1 + crescimento)
        )

        resultados.append({
            "cenario": cenario,
            "estado": estado,
            "demanda_base": demanda_base[estado],
            "crescimento_percentual": crescimento * 100,
            "demanda_projetada": round(demanda_projetada)
        })

df_cenarios = pd.DataFrame(resultados)

display(df_cenarios)

In [ ]:
import pandas as pd

query_custos = """
SELECT
    cidade,
    custo_entrada,
    custo_saida,
    custo_fixo,
    custo_total,
    classificacao
FROM raw.resultado_otimizacao
ORDER BY classificacao;
"""

df_custos = pd.read_sql(query_custos, engine)

display(df_custos)

In [ ]:
query_tabelas = """
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'raw'
ORDER BY table_name;
"""

df_tabelas = pd.read_sql(query_tabelas, engine)

display(df_tabelas)

In [ ]:
query_colunas = """
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'raw'
  AND table_name IN (
      'clientes',
      'pedidos',
      'itens_pedidos',
      'vendedores',
      'geo_cep',
      'centros_candidatos'
  )
ORDER BY table_name, ordinal_position;
"""

df_colunas = pd.read_sql(query_colunas, engine)

display(df_colunas)

In [ ]:
query_logistica = """
SELECT
    p.order_id,
    p.order_purchase_timestamp::timestamp AS data_compra,
    c.customer_state AS estado_cliente,
    c.customer_zip_code_prefix AS cep_cliente,
    v.seller_state AS estado_vendedor,
    v.seller_zip_code_prefix AS cep_vendedor,
    i.order_item_id,
    i.price,
    i.freight_value

FROM raw.pedidos p

INNER JOIN raw.clientes c
    ON p.customer_id = c.customer_id

INNER JOIN raw.itens_pedidos i
    ON p.order_id = i.order_id

INNER JOIN raw.vendedores v
    ON i.seller_id = v.seller_id

WHERE p.order_status = 'delivered'
    AND p.order_purchase_timestamp::timestamp
        >= DATE '2018-01-01'
    AND p.order_purchase_timestamp::timestamp
        < DATE '2018-09-01'
    AND c.customer_state IN ('SP', 'RJ', 'MG');
"""

df_logistica = pd.read_sql(query_logistica, engine)

print("Total de itens:", len(df_logistica))

display(df_logistica.head())

In [ ]:
query_coordenadas = """
SELECT
    i.order_id,
    i.order_item_id,
    c.customer_state AS estado_cliente,

    gc.latitude AS latitude_cliente,
    gc.longitude AS longitude_cliente,

    gv.latitude AS latitude_vendedor,
    gv.longitude AS longitude_vendedor

FROM raw.pedidos p

INNER JOIN raw.clientes c
    ON p.customer_id = c.customer_id

INNER JOIN raw.itens_pedidos i
    ON p.order_id = i.order_id

INNER JOIN raw.vendedores v
    ON i.seller_id = v.seller_id

LEFT JOIN raw.geo_cep gc
    ON c.customer_zip_code_prefix = gc.cep

LEFT JOIN raw.geo_cep gv
    ON v.seller_zip_code_prefix = gv.cep

WHERE p.order_status = 'delivered'
    AND p.order_purchase_timestamp::timestamp
        >= DATE '2018-01-01'
    AND p.order_purchase_timestamp::timestamp
        < DATE '2018-09-01'
    AND c.customer_state IN ('SP', 'RJ', 'MG');
"""

df_geo = pd.read_sql(query_coordenadas, engine)

# Verificar a disponibilidade das coordenadas
colunas_cliente = [
    "latitude_cliente",
    "longitude_cliente"
]

colunas_vendedor = [
    "latitude_vendedor",
    "longitude_vendedor"
]

cliente_valido = df_geo[colunas_cliente].notna().all(axis=1)
vendedor_valido = df_geo[colunas_vendedor].notna().all(axis=1)

ambos_validos = cliente_valido & vendedor_valido

print("Total de itens:", len(df_geo))
print("Itens com coordenadas do cliente:", cliente_valido.sum())
print("Itens com coordenadas do vendedor:", vendedor_valido.sum())
print("Itens com ambas as coordenadas:", ambos_validos.sum())

print(
    "Cobertura geográfica:",
    round(100 * ambos_validos.mean(), 2),
    "%"
)

In [ ]:
import numpy as np
import pandas as pd

# Manter apenas os itens com coordenadas completas
df_geo_valido = df_geo.dropna(
    subset=[
        "latitude_cliente",
        "longitude_cliente",
        "latitude_vendedor",
        "longitude_vendedor"
    ]
).copy()

# Buscar as coordenadas dos centros candidatos
query_centros = """
SELECT cidade, latitude, longitude
FROM raw.centros_candidatos;
"""

df_centros = pd.read_sql(query_centros, engine)

# Função para calcular distância geográfica
def haversine(lat1, lon1, lat2, lon2):
    raio_terra = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    diferenca_lat = lat2 - lat1
    diferenca_lon = lon2 - lon1

    a = (
        np.sin(diferenca_lat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(diferenca_lon / 2) ** 2
    )

    return 2 * raio_terra * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

# Calcular distâncias para cada centro candidato
resultados_distancias = []

for _, centro in df_centros.iterrows():

    dados = df_geo_valido.copy()

    dados["cidade_centro"] = centro["cidade"]

    # Vendedor até o centro
    dados["distancia_entrada_km"] = haversine(
        dados["latitude_vendedor"],
        dados["longitude_vendedor"],
        centro["latitude"],
        centro["longitude"]
    )

    # Centro até o cliente
    dados["distancia_saida_km"] = haversine(
        centro["latitude"],
        centro["longitude"],
        dados["latitude_cliente"],
        dados["longitude_cliente"]
    )

    resultados_distancias.append(dados)

df_distancias = pd.concat(
    resultados_distancias,
    ignore_index=True
)

print("Total de trajetos calculados:", len(df_distancias))

display(
    df_distancias[
        [
            "cidade_centro",
            "estado_cliente",
            "distancia_entrada_km",
            "distancia_saida_km"
        ]
    ].head()
)

In [ ]:
import pandas as pd

# Tarifas hipotéticas por item-quilômetro
tarifa_entrada = 0.08
tarifa_saida = 0.12

# Custos fixos hipotéticos para o período analisado
custos_fixos = {
    "guarulhos": 300000,
    "sao paulo": 350000,
    "santo andre": 280000,
    "campinas": 250000
}

# Taxas de crescimento por cenário e estado
taxas = df_cenarios[
    ["cenario", "estado", "crescimento_percentual"]
].copy()

taxas["fator_crescimento"] = (
    1 + taxas["crescimento_percentual"] / 100
)

# Associar cada trajeto aos cenários
simulacao = df_distancias.merge(
    taxas,
    left_on="estado_cliente",
    right_on="estado",
    how="inner",
    validate="many_to_many"
)

# Calcular custos ponderados pelo crescimento
simulacao["custo_entrada"] = (
    simulacao["distancia_entrada_km"]
    * tarifa_entrada
    * simulacao["fator_crescimento"]
)

simulacao["custo_saida"] = (
    simulacao["distancia_saida_km"]
    * tarifa_saida
    * simulacao["fator_crescimento"]
)

# Somar os custos por cenário e centro
resultado_cenarios = (
    simulacao
    .groupby(["cenario", "cidade_centro"], as_index=False)
    [["custo_entrada", "custo_saida"]]
    .sum()
)

# Adicionar os custos fixos
resultado_cenarios["custo_fixo"] = (
    resultado_cenarios["cidade_centro"]
    .map(custos_fixos)
)

# Calcular custo total
resultado_cenarios["custo_total"] = (
    resultado_cenarios["custo_entrada"]
    + resultado_cenarios["custo_saida"]
    + resultado_cenarios["custo_fixo"]
)

# Classificar as cidades dentro de cada cenário
resultado_cenarios["classificacao"] = (
    resultado_cenarios
    .groupby("cenario")["custo_total"]
    .rank(method="min")
    .astype(int)
)

# Ordenar os resultados
resultado_cenarios = resultado_cenarios.sort_values(
    ["cenario", "classificacao"]
)

display(resultado_cenarios.round(2))

In [ ]:
# Selecionar o cenário moderado
moderado = resultado_cenarios[
    resultado_cenarios["cenario"] == "Moderado"
].copy()

# Organizar por custo total
moderado = moderado.sort_values("custo_total")

# Identificar a melhor e a segunda melhor cidade
melhor = moderado.iloc[0]
segunda = moderado.iloc[1]

# Diferença entre os custos
diferenca = (
    segunda["custo_total"]
    - melhor["custo_total"]
)

# Limite do custo fixo da melhor cidade
limite_custo_fixo = (
    melhor["custo_fixo"] + diferenca
)

print("Melhor cidade:", melhor["cidade_centro"])
print("Segunda melhor:", segunda["cidade_centro"])

print(
    "Vantagem econômica: R$",
    round(diferenca, 2)
)

print(
    "Custo fixo máximo para manter a liderança: R$",
    round(limite_custo_fixo, 2)
)

In [ ]:
# Salvar resultados no PostgreSQL
resultado_cenarios.to_sql(
    name="resultado_cenarios",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

print("Resultados dos cenários salvos com sucesso!")

## Simulação de cenários de expansão da demanda

### Objetivo

Avaliar como diferentes hipóteses de crescimento da demanda
podem influenciar os custos logísticos e a escolha da
localização de um centro de distribuição.

### Período de referência

Foram utilizados os pedidos históricos entregues entre
janeiro e agosto de 2018, nos estados de SP, RJ e MG.

| Estado | Pedidos no período |
|--------|-------------------:|
| SP     | 23.335             |
| RJ     | 6.342              |
| MG     | 6.079              |

### Cenários hipotéticos

Foram definidos três cenários:

| Cenário     | SP  | RJ  | MG  |
|-------------|----:|----:|----:|
| Conservador | 5%  | 5%  | 5%  |
| Moderado    | 15% | 10% | 12% |
| Acelerado   | 30% | 20% | 25% |

Os percentuais representam hipóteses de crescimento
definidas para a análise de sensibilidade.

Não são previsões produzidas por Machine Learning.

### Aplicação na análise logística

As taxas de crescimento foram utilizadas para ajustar
a demanda considerada na simulação.

O modelo assume que a quantidade de itens transportados
cresce proporcionalmente à demanda de pedidos em cada estado.

Para cada cenário, foram estimados os custos logísticos
das quatro cidades candidatas:

- Guarulhos
- São Paulo
- Santo André
- Campinas

A localização recomendada corresponde à alternativa
com menor custo total estimado dentro das premissas
estabelecidas.

### Limitações

Os resultados dependem das taxas hipotéticas de crescimento,
dos custos unitários de transporte e dos custos fixos
considerados na simulação.

A análise não incorpora tarifas reais de frete,
distâncias rodoviárias ou restrições operacionais
dos centros de distribuição.

## Otimização da localização do centro de distribuição

### 1. Objetivo

Identificar a localização com menor custo logístico estimado
entre quatro cidades candidatas:

- Guarulhos
- São Paulo
- Santo André
- Campinas

A análise considera os deslocamentos entre vendedores,
centros de distribuição e consumidores.

### 2. Metodologia

Foram utilizadas informações históricas da plataforma
Olist (2016–2018), incluindo:

- Localização geográfica dos vendedores.
- Localização geográfica dos consumidores.
- Quantidade de itens transportados.
- Coordenadas dos centros de distribuição candidatos.

As distâncias foram estimadas utilizando a fórmula de
Haversine, que calcula a distância aproximada entre
dois pontos geográficos a partir de suas coordenadas.

As distâncias representam trajetos em linha geodésica,
não percursos rodoviários reais.

### 3. Premissas de custos

Foram definidos os seguintes valores hipotéticos:

| Componente | Custo |
|------------|-------|
| Transporte de entrada | R$ 0,08 por item/km |
| Transporte de saída | R$ 0,12 por item/km |
| Custo fixo | Varia conforme a cidade |

O custo logístico total foi calculado como:

**Custo total = Custo de entrada + Custo de saída + Custo fixo**

### 4. Resultados da otimização histórica

| Classificação | Cidade | Custo total estimado |
|---------------|--------|---------------------:|
| 1º | Guarulhos | R$ 9.266.654,62 |
| 2º | São Paulo | R$ 9.327.342,95 |
| 3º | Santo André | R$ 9.360.761,63 |
| 4º | Campinas | R$ 9.363.139,90 |

Guarulhos apresentou o menor custo total estimado.

A diferença entre Guarulhos e São Paulo,
segunda colocada, foi de R$ 60.688,33.

### 5. Resultados da simulação de cenários

A análise também avaliou os cenários Conservador,
Moderado e Acelerado.

Guarulhos permaneceu como a alternativa de menor
custo estimado nos três cenários.

No cenário Moderado:

| Indicador | Resultado |
|-----------|-----------|
| Melhor localização | Guarulhos |
| Custo estimado de Guarulhos | R$ 2.285.142,73 |
| Segunda melhor localização | Santo André |
| Custo estimado de Santo André | R$ 2.316.375,25 |
| Economia estimada | R$ 31.232,52 |

Os valores da otimização histórica e da simulação
de cenários correspondem a análises com bases e
premissas de demanda distintas e não devem ser
comparados diretamente como se fossem o mesmo período.

### 6. Conclusão

Dentro das premissas adotadas, Guarulhos apresentou
o menor custo logístico estimado.

O resultado indica que a cidade merece prioridade
em uma avaliação mais detalhada de expansão.

Entretanto, a escolha definitiva exigiria informações
adicionais sobre:

- Custos reais de transporte.
- Distâncias e condições das rotas rodoviárias.
- Capacidade operacional dos centros de distribuição.
- Disponibilidade e custos de instalações.
- Prazos de entrega e níveis de serviço.

Portanto, a recomendação representa o resultado de
uma simulação quantitativa, não uma decisão definitiva
de implantação.